<a href="https://colab.research.google.com/github/austraraptor/2025_UNAP/blob/main/Lab_Imagenes_en_paralelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 4: Procesamiento de imágenes en paralelo
**Computación Paralela 2026-II, Universidad Nacional del Altiplano**
**Docente:** M.Sc. Alain Paul Herrera Urtiaga

**Nombre:** ____________________ **Código:** ____________



Donde veas `COMPLETAR`, reemplázalo por tu código.

In [ ]:
# Herramientas del laboratorio (solo ejecuta esta celda)
import os, time, glob
import numpy as np
import cv2
import matplotlib.pyplot as plt
from multiprocessing import Pool

cv2.setNumThreads(1)   # OpenCV usa varios hilos por su cuenta: lo apagamos para medir solo nuestro paralelismo

class _Completar:
    def __repr__(self): return "COMPLETAR"
    def _propaga(self, *a): return self
    __add__ = __radd__ = __sub__ = __rsub__ = __mul__ = __rmul__ = _propaga
    __truediv__ = __rtruediv__ = __pow__ = __rpow__ = __neg__ = __getitem__ = _propaga
    def astype(self, *a, **k): return self
COMPLETAR = _Completar()

def _tiene_completar(x):
    if isinstance(x, _Completar): return True
    if isinstance(x, (list, tuple)): return any(_tiene_completar(v) for v in x)
    return False

def falta(nombre):
    print(f" {nombre}: falta completar. Reemplaza COMPLETAR por tu código y vuelve a ejecutar.")

def mostrar(imagenes, titulos):
    """Muestra varias imágenes en una fila."""
    if any(_tiene_completar(im) for im in imagenes):
        falta("Filtros"); return
    fig, ejes = plt.subplots(1, len(imagenes), figsize=(4 * len(imagenes), 4))
    for eje, im, t in zip(np.atleast_1d(ejes), imagenes, titulos):
        eje.imshow(im, cmap="gray" if im.ndim == 2 else None)
        eje.set_title(t); eje.axis("off")
    plt.tight_layout(); plt.show()

def imagen_ejemplo():
    """Paisaje de prueba por si no subes foto."""
    alto, ancho = 600, 900
    y = np.linspace(0, 1, alto)[:, None]
    cielo = np.dstack([90 + 120 * y + 0 * np.ones((1, ancho)), 150 + 80 * y + 0 * np.ones((1, ancho)), 230 - 30 * y + 0 * np.ones((1, ancho))])
    img = cielo.astype(np.uint8)
    cv2.circle(img, (720, 130), 60, (255, 230, 120), -1)
    pts = np.array([[0, 600], [0, 380], [200, 220], [380, 360], [560, 180], [900, 400], [900, 600]])
    cv2.fillPoly(img, [pts], (70, 110, 60))
    cv2.rectangle(img, (0, 500), (900, 600), (40, 120, 170), -1)
    return img

print(" Listo. Núcleos disponibles:", os.cpu_count())

---
## Parte 1: Una imagen es una matriz
Ejecuta la celda y **elige una foto tuya** (jpg o png). Si cancelas, se usa un paisaje de prueba.

In [ ]:
foto = None
try:
    from google.colab import files
    subido = files.upload()
    if subido:
        nombre = next(iter(subido))
        foto = cv2.cvtColor(cv2.imread(nombre), cv2.COLOR_BGR2RGB)   # OpenCV lee en BGR
except Exception:
    pass
if foto is None:
    foto = imagen_ejemplo()

# Reducimos fotos muy grandes para que todo vaya rápido
if foto.shape[1] > 1000:
    escala = 1000 / foto.shape[1]
    foto = cv2.resize(foto, None, fx=escala, fy=escala)

print("Forma (alto, ancho, canales):", foto.shape)
print("Tipo de dato:", foto.dtype, "  valores de 0 a 255")
print("Píxel del centro (R, G, B):", foto[foto.shape[0] // 2, foto.shape[1] // 2])
mostrar([foto], ["Tu foto"])

**Pregunta 1.** Si tu foto mide 1000 × 750 píxeles, ¿cuántos números guarda en total?

*Respuesta:*

---
## Parte 2: Filtros
Un filtro es un **lazo DOALL** sobre los píxeles: cada píxel nuevo depende solo del píxel original. Por eso se puede **vectorizar** con NumPy, igual que el Ejercicio 4 de la Sesión 3.

**Escala de grises:** `gris = 0.299 R + 0.587 G + 0.114 B`

Primero mira la versión con lazos (solo en un recorte de 100 × 100, porque es lenta):

In [ ]:
recorte = foto[:100, :100].astype(float)

t0 = time.perf_counter()
gris_for = np.zeros((100, 100))
for i in range(100):
    for j in range(100):
        r, g, b = recorte[i, j]
        gris_for[i, j] = 0.299 * r + 0.587 * g + 0.114 * b
t_for = time.perf_counter() - t0
print(f"Con lazos, 10 000 píxeles: {t_for*1000:.1f} ms")

**Tu tarea:** escribe la versión vectorizada, sin lazos. Usa `R`, `G` y `B`.

In [ ]:
R, G, B = foto[:, :, 0].astype(float), foto[:, :, 1].astype(float), foto[:, :, 2].astype(float)

t0 = time.perf_counter()
gris = COMPLETAR                       # fórmula de grises con R, G y B
t_np = time.perf_counter() - t0

if _tiene_completar(gris):
    falta("Escala de grises")
else:
    gris = gris.astype(np.uint8)
    print(f"Con NumPy, {gris.size:,} píxeles: {t_np*1000:.1f} ms")

Los otros tres filtros ya están listos. **Sepia** es otra fórmula por píxel; **bordes** y **desenfoque** usan los vecinos de cada píxel, por eso cuestan más.

In [ ]:
M_SEPIA = np.array([[0.393, 0.769, 0.189],
                    [0.349, 0.686, 0.168],
                    [0.272, 0.534, 0.131]])
sepia = np.clip(foto.astype(float) @ M_SEPIA.T, 0, 255).astype(np.uint8)

if _tiene_completar(gris):
    falta("Escala de grises")
else:
    bordes = cv2.Canny(gris, 100, 200)
    desenfoque = cv2.GaussianBlur(foto, (31, 31), 0)
    mostrar([foto, gris, sepia], ["Original", "Grises", "Sepia"])
    mostrar([bordes, desenfoque], ["Bordes", "Desenfoque"])

**Pregunta 2.** ¿Cuántas veces más rápida fue la versión NumPy por píxel? Compara: tiempo con lazos / 10 000 contra tiempo NumPy / total de píxeles.

*Respuesta:*

---
## Parte 3: Contar objetos en 50 imágenes
Simulamos 50 fotos de un campo con manchas (por ejemplo, plantas vistas desde un dron). Para cada imagen hay que **contar cuántas manchas tiene**:

1. Leer la imagen del disco
2. Pasarla a grises y desenfocarla para quitar el ruido
3. Umbralizar: separar manchas del fondo
4. Contar los contornos

Cada imagen se procesa sola, sin mirar a las demás: **es un lazo DOALL sobre las imágenes.**

Primero generamos las 50 imágenes y guardamos cuántas manchas tiene cada una, para verificar después.

In [ ]:
os.makedirs("campo", exist_ok=True)
gen = np.random.default_rng(2026)
reales = {}
for k in range(50):
    alto, ancho = 1200, 1600
    img = gen.integers(60, 110, (alto, ancho, 3)).astype(np.uint8)     # suelo con ruido
    n = int(gen.integers(8, 25))
    centros = []
    while len(centros) < n:                                              # manchas sin tocarse
        c = (int(gen.integers(60, ancho - 60)), int(gen.integers(60, alto - 60)))
        if all((c[0] - x) ** 2 + (c[1] - y) ** 2 > 130 ** 2 for x, y in centros):
            centros.append(c)
    for c in centros:
        cv2.circle(img, c, int(gen.integers(25, 50)), (40, int(gen.integers(160, 230)), 40), -1)
    ruta = f"campo/img_{k:02d}.png"
    cv2.imwrite(ruta, img)
    reales[ruta] = n

rutas = sorted(reales)
print("Imágenes creadas:", len(rutas))
ejemplo = cv2.cvtColor(cv2.imread(rutas[0]), cv2.COLOR_BGR2RGB)
mostrar([ejemplo], [f"{rutas[0]}: {reales[rutas[0]]} manchas"])

La función que procesa **una** imagen:

In [ ]:
def contar_objetos(ruta):
    img = cv2.imread(ruta)
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    suave = cv2.GaussianBlur(gris, (21, 21), 0)
    _, mascara = cv2.threshold(suave, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mascara = cv2.morphologyEx(mascara, cv2.MORPH_OPEN, np.ones((7, 7), np.uint8))
    contornos, _ = cv2.findContours(mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return len([c for c in contornos if cv2.contourArea(c) > 500])

print("Manchas detectadas en la primera imagen:", contar_objetos(rutas[0]), " reales:", reales[rutas[0]])

**Versión secuencial:** un `for` sobre las 50 imágenes.

In [ ]:
t0 = time.perf_counter()
conteos_seq = [contar_objetos(r) for r in rutas]
t_seq = time.perf_counter() - t0

aciertos = sum(c == reales[r] for c, r in zip(conteos_seq, rutas))
print(f"Secuencial: {t_seq:.2f} s   aciertos: {aciertos}/50")

**Tu tarea:** reparte las 50 imágenes con `Pool.map`. ¿Qué función? ¿Sobre qué datos?

In [ ]:
funcion = COMPLETAR
datos   = COMPLETAR

if _tiene_completar([funcion, datos]):
    falta("Pool.map")
else:
    t0 = time.perf_counter()
    with Pool(2) as pool:
        conteos_par = pool.map(funcion, datos)
    t_par = time.perf_counter() - t0
    print(f"Paralelo con 2 procesos: {t_par:.2f} s")
    print("✅ Mismos conteos" if conteos_par == conteos_seq else "❌ Los conteos cambiaron")
    print(f"Speedup: {t_seq / t_par:.2f}")

**Pregunta 3.** En la Sesión 3, `Pool` fue más lento que el `for`. ¿Por qué aquí sí gana?

*Respuesta:*

**Pregunta 4.** Según la taxonomía de Flynn, ¿qué tipo de paralelismo es repartir las imágenes entre procesos? ¿Y el filtro de grises con NumPy?

*Respuesta:*

---
## Parte 4: Speedup y eficiencia
Medimos con 1, 2 y 4 procesos.

- **Speedup** = tiempo secuencial / tiempo paralelo
- **Eficiencia** = speedup / número de procesos (1.0 sería perfecto)

**Tu tarea:** completa las dos fórmulas.

In [ ]:
procesos = [1, 2, 4]
tiempos, speedups, eficiencias = [], [], []
for p in procesos:
    t0 = time.perf_counter()
    with Pool(p) as pool:
        pool.map(contar_objetos, rutas)
    t = time.perf_counter() - t0
    s = COMPLETAR              # speedup
    e = COMPLETAR              # eficiencia
    tiempos.append(t); speedups.append(s); eficiencias.append(e)

if _tiene_completar(speedups + eficiencias):
    falta("Speedup y eficiencia")
else:
    print(f"{'Procesos':>9} {'Tiempo (s)':>11} {'Speedup':>8} {'Eficiencia':>11}")
    for p, t, s, e in zip(procesos, tiempos, speedups, eficiencias):
        print(f"{p:>9} {t:>11.2f} {s:>8.2f} {e:>11.2f}")
    plt.figure(figsize=(6, 4))
    plt.plot(procesos, procesos, "--", color="gray", label="Ideal")
    plt.plot(procesos, speedups, "o-", label="Medido")
    plt.xlabel("Procesos"); plt.ylabel("Speedup"); plt.xticks(procesos)
    plt.title("Speedup al contar objetos en 50 imágenes"); plt.legend(); plt.grid(alpha=0.3); plt.show()

**Pregunta 5.** Colab gratuito tiene 2 núcleos. ¿Qué pasó con 4 procesos? ¿Por qué?

*Respuesta:*

**Pregunta 6.** ¿Por qué con 2 procesos el speedup no llega exactamente a 2?

*Respuesta:*

---
### Reto opcional
Cambia el tamaño del desenfoque en `contar_objetos` de `(21, 21)` a `(5, 5)`, vuelve a ejecutar las Partes 3 y 4 y compara el speedup. ¿Qué pasa cuando cada tarea tiene menos cálculo?